##### Imports

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML

from gg570_d200.auxiliary_functions.forest_riesz_funcs import call_forestriesz, call_forestriesz_cross, calculate_p_value
from gg570_d200.auxiliary_functions.ate_estimation_funcs import forest_riesz_gate, forest_riesz_gate_cross, causal_dml_gate

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
root = Path.cwd().parent
data_path = root / "data"

In [3]:
np.random.seed(21)

In [4]:
df_scaled = pd.read_csv(data_path / "df_scaled.csv")

In [5]:
saved_joblib = joblib.load(data_path / "covariate_scaler.joblib")
scaler = saved_joblib['scaler']

In [6]:
df = pd.read_csv(data_path / "df.csv")

In [7]:
treatment_col = 'Training (last 3 months)'
outcome_col = 'Underemployment hours'
covariate_cols = [col for col in df_scaled.columns if col not in treatment_col and col not in outcome_col and col not in ['prop_scores']]

In [8]:
relevant_gates = set()

In [9]:
def add_relevant_gates(est):
    global relevant_gates
    if isinstance(est, list):
        for est_i in est:
            heterogeneity_importance = est_i.feature_importances_
            top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
            for id in top_3_ids:
                relevant_gates.add(id)
    else:
        heterogeneity_importance = est.feature_importances_
        top_3_ids = np.argsort(heterogeneity_importance)[-3:][::-1]
        for id in top_3_ids:
            relevant_gates.add(id)

---

##### Key specification: ForestRiesz (cross-validated, scaled)

In [10]:
cross_scale, est_cs, test_id_cs = call_forestriesz_cross(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_cs)
cross_scale

{'dr': {'est': -0.1693296470300766,
  'low': -1.8117341606998068,
  'high': 1.4730748666396534,
  'p_val': 0.8398623196865482},
 'plugin': {'est': -0.03742179387258689,
  'low': -0.5965905370826727,
  'high': 0.5217469493374989,
  'p_val': 0.895641953248558}}

---

##### Alternative specifications: ForestRiesz (not cross-validated, scaled)

In [11]:
no_cross_scale, est_ncs = call_forestriesz(df_scaled, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_ncs)
no_cross_scale

{'dr': {'est': -0.2665091577031718,
  'low': -0.8744295027331255,
  'high': 0.34141118732678194,
  'p_val': 0.39020914137380824},
 'plugin': {'est': -0.03658136219478732,
  'low': -0.2950668292160727,
  'high': 0.22190410482649808,
  'p_val': 0.7814899601055587}}

---

##### Alternative specifications: ForestRiesz (not cross-validated, not scaled)

In [12]:
no_cross_no_scale, est_ncns = call_forestriesz(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_ncns)
no_cross_no_scale

{'dr': {'est': -0.26791976018597563,
  'low': -0.8760499672492761,
  'high': 0.3402104468773248,
  'p_val': 0.3878692645615842},
 'plugin': {'est': -0.03657351864532722,
  'low': -0.2951955071691021,
  'high': 0.22204846987844767,
  'p_val': 0.7816480220116433}}

---

##### Alternative specifications: ForestRiesz (cross-validated, not scaled)

In [13]:
cross_no_scale, est_cns, test_id_cns = call_forestriesz_cross(df, covariate_cols, treatment_col, outcome_col, methods=['dr', 'plugin'], return_est=True)
add_relevant_gates(est_cns)
cross_no_scale

{'dr': {'est': -0.16930556883926598,
  'low': -1.806674362019337,
  'high': 1.4680632243408052,
  'p_val': 0.8393990435692609},
 'plugin': {'est': -0.038289273524296126,
  'low': -0.5974631242549585,
  'high': 0.5208845772063663,
  'p_val': 0.8932381100393474}}

---

##### Alternative specifications: CausalForestDML (cross-validated, scaled)

In [14]:
folds = KFold(n_splits=3, shuffle=True, random_state=21)

In [15]:
test_id_cdml = [x[1] for x in folds.split(df[covariate_cols].values)]

In [16]:
causal_dml = CausalForestDML(
    discrete_treatment=True,
    criterion='mse',
    n_estimators=100,
    min_samples_leaf=2,
    min_var_fraction_leaf=0.001,
    min_var_leaf_on_val=True,
    min_impurity_decrease=0.01,
    max_depth=None,
    inference=True,
    subforest_size=2,
    honest=True,
    verbose=0,
    n_jobs=-2,
    random_state=21,
    cv=folds
)

In [17]:
causal_dml.fit(df_scaled[outcome_col], df_scaled[treatment_col], X=df_scaled[covariate_cols])
add_relevant_gates(causal_dml)
cdml_ate, cdml_low, cdml_high = causal_dml.ate_[0], (causal_dml.ate_-1.96*causal_dml.ate_stderr_)[0], (causal_dml.ate_+1.96*causal_dml.ate_stderr_)[0]
cdml_p_val = calculate_p_value(cdml_ate, cdml_low, cdml_high)
cdml_ate, cdml_low, cdml_high, cdml_p_val

---

##### Saving ATE results

In [18]:
ate_results_df = pd.DataFrame(columns=['Method', 'Point estimate', 'Low (95% CI)', 'High (95% CI)', 'p value'])

for est in (["FR_DR_CS", cross_scale], ["FR_DR_NCS", no_cross_scale], ["FR_DR_NCNS", no_cross_no_scale], ["FR_DR_CNS", cross_no_scale]):
    ate_results_df.loc[len(ate_results_df)] = [est[0]] + list(est[1]['dr'].values())

ate_results_df.loc[len(ate_results_df)] = ["CausalDML", cdml_ate, cdml_low, cdml_high, cdml_p_val]

In [19]:
ate_results_df

,Method,Point estimate,Low (95% CI),High (95% CI),p value
0,FR_DR_CS,-0.169330,-1.811734,1.473075,0.839862
1,FR_DR_NCS,-0.266509,-0.874430,0.341411,0.390209
2,FR_DR_NCNS,-0.267920,-0.876050,0.340210,0.387869
3,FR_DR_CNS,-0.169306,-1.806674,1.468063,0.839399
4,CausalDML,0.155675,-0.738675,1.050026,0.732982


In [20]:
(root / "results").mkdir(parents=True, exist_ok=True)
ate_results_df.to_csv(root / f"results/ate_results.csv", index=False)

---

##### Relevant Group ATEs (GATEs)

In [21]:
relevant_gates = sorted(relevant_gates)

In [22]:
for i in relevant_gates:
    print(covariate_cols[i], df_scaled[covariate_cols[i]].nunique())

AGE 48
FTPTWK 2
HIQUAL22 39
INECAC05 2
LKWFWM_bin_(12.2, 15.0] 2
LKWFWM_bin_(6.6, 9.4] 2
REFWKD 31


In [36]:
gate_results_df = pd.DataFrame(columns=['GATE', 'Method', 'Point estimate', 'Low (95% CI)', 'High (95% CI)', 'p value'])

In [37]:
for i in relevant_gates:
    if df[covariate_cols[i]].nunique() == 2:
        mask = lambda df, i: df[covariate_cols[i]]==max(df[covariate_cols[i]]) if mask_status else df[covariate_cols[i]] < max(df[covariate_cols[i]])

        for status in [True, False]:
            mask_status = status
            gate_label = f"{covariate_cols[i]}_{status}"
            
            for method_name, result in [
                ("FR_DR_CS", forest_riesz_gate_cross(df_scaled, covariate_cols, treatment_col, outcome_col, est_cs, test_id_cs, mask(df_scaled, i))),
                ("FR_DR_NCS", forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, mask(df_scaled, i))),
                ("FR_DR_NCNS", forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask(df, i))),
                ("FR_DR_CNS", forest_riesz_gate_cross(df, covariate_cols, treatment_col, outcome_col, est_cns, test_id_cns, mask(df, i))),
                ("CausalDML", causal_dml_gate(df_scaled, covariate_cols, treatment_col, outcome_col, causal_dml, test_id_cdml, mask(df_scaled, i)))
            ]:
                gate_results_df.loc[len(gate_results_df)] = [gate_label, method_name] + result

In [38]:
for i in relevant_gates:
    if df[covariate_cols[i]].nunique() > 2:
        bins = pd.qcut(df[covariate_cols[i]], q=5, duplicates='drop')
        
        for bin_label in bins.unique():
            mask = bins == bin_label
            gate_label = f"{covariate_cols[i]}_{bin_label}"
            
            for method_name, result in [
                ("FR_DR_CS", forest_riesz_gate_cross(df_scaled, covariate_cols, treatment_col, outcome_col, est_cs, test_id_cs, mask)),
                ("FR_DR_NCS", forest_riesz_gate(df_scaled, covariate_cols, treatment_col, outcome_col, est_ncs, mask)),
                ("FR_DR_NCNS", forest_riesz_gate(df, covariate_cols, treatment_col, outcome_col, est_ncns, mask)),
                ("FR_DR_CNS", forest_riesz_gate_cross(df, covariate_cols, treatment_col, outcome_col, est_cns, test_id_cns, mask)),
                ("CausalDML", causal_dml_gate(df_scaled, covariate_cols, treatment_col, outcome_col, causal_dml, test_id_cdml, mask))
            ]:
                gate_results_df.loc[len(gate_results_df)] = [gate_label, method_name] + result

In [40]:
gate_results_df.sort_values(by='p value').head(10)

,GATE,Method,Point estimate,Low (95% CI),High (95% CI),p value
2,FTPTWK_True,FR_DR_NCNS,-0.985785,-1.768851,-0.202719,0.013611
1,FTPTWK_True,FR_DR_NCS,-0.985081,-1.767979,-0.202182,0.013658
67,"HIQUAL22_(0.999, 8.0]",FR_DR_NCNS,-0.887557,-1.896390,0.121275,0.084645
66,"HIQUAL22_(0.999, 8.0]",FR_DR_NCS,-0.883917,-1.892490,0.124655,0.085848
52,"AGE_(16.999, 31.0]",FR_DR_NCNS,-1.012419,-2.189044,0.164207,0.091712
51,"AGE_(16.999, 31.0]",FR_DR_NCS,-1.007770,-2.183992,0.168452,0.093100
56,"AGE_(31.0, 38.0]",FR_DR_NCS,0.970049,-0.348534,2.288632,0.149331
57,"AGE_(31.0, 38.0]",FR_DR_NCNS,0.968841,-0.350707,2.288388,0.150137
41,"AGE_(38.0, 45.0]",FR_DR_NCS,-0.845954,-2.081347,0.389439,0.179559
42,"AGE_(38.0, 45.0]",FR_DR_NCNS,-0.846165,-2.081997,0.389668,0.179605


In [41]:
(root / "results").mkdir(parents=True, exist_ok=True)
gate_results_df.to_csv(root / f"results/gate_results.csv", index=False)